# Tendril bounded benchmark
Runs on either notebook lane. Exercises scientific Python, IPython, plots, process pools, and small artifacts. No external services required. Increase workload sizes only within your node's memory, runtime, and prepaid credit. Downloads stay below 4 MB total; large local scratch files are not returned.

In [ ]:
%pip --version
import sys, time, json, hashlib, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn
import psutil
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
rng = np.random.default_rng(42)
summary = {}
Path('results').mkdir(exist_ok=True)
print({'python': sys.version.split()[0], 'numpy': np.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__})
print('RSS MB:', round(psutil.Process().memory_info().rss / 1e6, 1))

In [ ]:
started = time.perf_counter()
a = rng.normal(size=(600, 600))
b = a @ a.T
summary['matrix_seconds'] = time.perf_counter() - started
print('Matrix checksum:', float(np.trace(b)))
print('Matrix seconds:', round(summary['matrix_seconds'], 3))

In [ ]:
started = time.perf_counter()
df = pd.DataFrame({'group': rng.integers(0, 20, 100000), 'value': rng.normal(size=100000)})
groups = df.groupby('group')['value'].agg(['mean', 'std', 'count'])
groups.to_csv('results/groups.csv')
summary['pandas_seconds'] = time.perf_counter() - started
print(groups.head())
groups['mean'].plot.bar(title='Group means')
plt.tight_layout()
plt.savefig('results/groups.png', dpi=100)
plt.show()

In [ ]:
started = time.perf_counter()
X, y = make_classification(n_samples=3000, n_features=20, random_state=42)
model = RandomForestClassifier(n_estimators=30, max_depth=8, n_jobs=2, random_state=42)
model.fit(X, y)
summary['model_seconds'] = time.perf_counter() - started
summary['training_accuracy'] = float(model.score(X, y))
print('Training accuracy (smoke test, not held-out evaluation):', summary['training_accuracy'])

In [ ]:
from concurrent.futures import ProcessPoolExecutor
def is_prime(n):
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    for divisor in range(3, int(n ** 0.5) + 1, 2):
        if n % divisor == 0:
            return False
    return True
started = time.perf_counter()
nums = range(100000, 105000)
with ProcessPoolExecutor(max_workers=2) as ex:
    summary['prime_count'] = sum(ex.map(is_prime, nums, chunksize=100))
summary['process_pool_seconds'] = time.perf_counter() - started
print('Prime count:', summary['prime_count'])

In [ ]:
pixels = rng.integers(0, 256, (128, 128, 3), dtype=np.uint8)
Image.fromarray(pixels).save('results/sample.png')
blob = rng.bytes(64000)
Path('results/sample.bin').write_bytes(blob)
summary['sha256'] = hashlib.sha256(blob).hexdigest()
Path('results/summary.json').write_text(json.dumps(summary, indent=2))
with zipfile.ZipFile('results/bundle.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(Path('results').iterdir()):
        if path.name != 'bundle.zip':
            archive.write(path, arcname=path.name)
print(json.dumps(summary, indent=2))
print('Artifact bytes total:', sum(path.stat().st_size for path in Path('results').iterdir()))